In [7]:
import requests
import pandas as pd
import time
import re
import os
from datetime import datetime

PUBLIC_KEY = "37c04fe6-a560-4549-b459-02309cf643ad"
BASE_URL = "https://public-api.reviews.2gis.com/2.0/branches/{branch_id}/reviews"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json",
    "Referer": "https://2gis.ru/",
}

PLACES = [
    {"name": "Surf Coffee Сколково", "branch_id": "70000001065464766"},
    {"name": "Фудхолл Сколково",      "branch_id": "70000001049957601"},
    {"name": "Петрушка Сколково",     "branch_id": "70000001098535255"},
    {"name": "Кофемания Сколково",    "branch_id": "70000001109879701"},
    {"name": "Груша (РЭШ)",           "branch_id": "70000001062919413"},
]


OUTPUT_PATH = "data/2gis_reviews.csv"

PAIN_PATTERNS = {
    "очередь":  r"очередь|ждал|долго|медленн",
    "цены":     r"дорого|цена|стоимость",
    "качество": r"холодн|невкусн|плохо|разочаров",
    "тесно":    r"шумно|тесно|мест нет|не сесть",
    "wifi":     r"wifi|интернет|розетк",
    "сервис":   r"хам|невежлив|игнор|официант",
}

def fetch_reviews(branch_id, limit=50):
    url = BASE_URL.format(branch_id=branch_id)
    params = {
        "limit": limit,
        "is_advertiser": "false",
        "fields": "meta.branch_rating,meta.branch_reviews_count",
        "without_my_first_review": "false",
        "rated": "true",
        "sort_by": "date_edited",
        "key": PUBLIC_KEY,
    }
    try:
        response = requests.get(url, params=params, headers=HEADERS, timeout=15)
        response.raise_for_status()
        return response.json().get("reviews", [])
    except requests.RequestException as e:
        print(f"  ошибка {e}")
        return []
    except ValueError:
        print(f" невалидный JSON")
        return []

def parse_review(raw, place_name):
    user = raw.get("user") or {}
    return {
        "place":        place_name,
        "rating":       raw.get("rating"),
        "text":         (raw.get("text") or "").strip(),
        "date":         (raw.get("date_created") or "")[:10],
        "likes":        raw.get("likes_count", 0),
        "comments":     raw.get("comments_count", 0),
        "user_name":    user.get("name", "—"),
        "collected_at": datetime.now().strftime("%Y-%m-%d"),
    }

def extract_pain_keywords(text):
    t = text.lower()
    return [label for label, pat in PAIN_PATTERNS.items() if re.search(pat, t)]


os.makedirs("data", exist_ok=True)
all_reviews = []

for place in PLACES:
    print(f"{place['name']}")
    raw_reviews = fetch_reviews(place["branch_id"], limit=50)
    print(f"    Получено отзывов: {len(raw_reviews)}")
    for raw in raw_reviews:
        row = parse_review(raw, place["name"])
        row["pain_keywords"] = ", ".join(extract_pain_keywords(row["text"]))
        all_reviews.append(row)
    time.sleep(1)

columns = ["place","rating","text","date","likes","comments",
           "user_name","collected_at","pain_keywords"]

if not all_reviews:
    print("\nДанные не собраны")
    df = pd.DataFrame(columns=columns)
else:
    df = pd.DataFrame(all_reviews)
    stats = df.groupby("place")["rating"].agg(["count","mean"]).round(2)
    stats.columns = ["отзывов", "ср. рейтинг"]
    print("\nСтатистика:")
    print(stats.to_string())

df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"\n Итого: {len(df)} отзыва  {OUTPUT_PATH}")
df.head()

Surf Coffee Сколково
    Получено отзывов: 12
Фудхолл Сколково
    Получено отзывов: 35
Петрушка Сколково
    Получено отзывов: 2
Кофемания Сколково
    Получено отзывов: 3
Груша (РЭШ)
    Получено отзывов: 1

Статистика:
                      отзывов  ср. рейтинг
place                                     
Surf Coffee Сколково       12         4.92
Груша (РЭШ)                 1         5.00
Кофемания Сколково          3         5.00
Петрушка Сколково           2         4.00
Фудхолл Сколково           35         4.54

 Итого: 53 отзыва  data/2gis_reviews.csv


,place,rating,text,date,likes,comments,user_name,collected_at,pain_keywords
0,Surf Coffee Сколково,5,"Очень нравится эта точка🖤 баристы отзывчивые, ...",2025-12-04,0,0,Ilona Baturina,2026-05-24,
1,Surf Coffee Сколково,4,Раньше готовили вкуснее кофе. Либо помол не пр...,2025-08-03,0,2,Татьяна Константиновна,2026-05-24,качество
2,Surf Coffee Сколково,5,"Замечательная кофейня, сердечная благодарность...",2025-05-30,0,1,Марина Тленова,2026-05-24,
3,Surf Coffee Сколково,5,"Сёрф хорош, как всегда! Спасибо бариста за атм...",2025-02-25,0,0,Кирилл Шешин,2026-05-24,
4,Surf Coffee Сколково,5,Отличный повод прогуляться перед электричкой з...,2024-09-01,0,1,Андрей Дегтярев,2026-05-24,
